# Khipus.ai
## Módulo 5 · Unidad 1: introducción a la IA generativa y los LLM
### Demo: generación de texto con Microsoft Foundry

<span>© Copyright Notice 2025, Khipus.ai - All Rights Reserved.</span>

Adaptación a Microsoft Foundry: 8 de octubre de 2026.

## Objetivos
1. Conectar Python con el despliegue `gpt-4.1-mini` del proyecto existente.
2. Distinguir las instrucciones del sistema y la solicitud del usuario.
3. Comparar un prompt breve con uno que especifica público y formato.
4. Observar el consumo de tokens y el motivo de finalización.

Ejecuta las celdas en orden. La práctica realiza **dos solicitudes de chat**, facturadas en Azure. Las respuestas pueden variar entre ejecuciones.

## Antes de empezar: preparar el entorno virtual
Estos tres notebooks comparten el único archivo [`requirements.txt`](requirements.txt) de esta carpeta. En PowerShell, desde esta carpeta, crea el entorno si aún no existe y actívalo:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Si todavía no instalaste las dependencias (o cambió `requirements.txt`), instálalas una sola vez:

```powershell
python -m pip install -r requirements.txt
```

En VS Code selecciona `.venv` como intérprete y kernel. Si ya preparaste el entorno y los requirements, continúa sin repetir la instalación.

In [9]:
# EN CASO DE NO HABER CREADO EL .VENV
# %python -m venv .venv

# ACTIVAR EL ENTORNO VIRTUAL
# Windows
# .venv\Scripts\activate
# macOS / Linux
# source .venv/bin/activate

# INSTALAR LAS DEPENDENCIAS
# %pip install -r requirements.txt

## Configurar la conexión
Define los valores en la celda de constantes y ejecútala antes de crear el cliente:
- `AZURE_OPENAI_API_KEY`: clave del recurso proporcionada por el instructor.
- `AZURE_OPENAI_API_BASE`: endpoint Azure OpenAI terminado en `/openai/v1`.
- `AZURE_OPENAI_CHAT_DEPLOYMENT`: nombre del despliegue que usará esta práctica.

La conexión y las solicitudes toman sus valores de estas constantes. No se leen variables de entorno ni se solicita una contraseña interactiva.

La ruta **Azure OpenAI v1** ya incluye la versión; no se necesita `AZURE_OPENAI_API_VERSION`. El Project Endpoint (`/api/projects/...`) no se utiliza para estas llamadas. La configuración no envía solicitudes; la primera llamada a la API comprobará tus credenciales.

In [10]:
from urllib.parse import urlparse
from openai import OpenAI

In [11]:
AZURE_OPENAI_API_KEY = "YOUR_AZURE_API_KEY"
AZURE_OPENAI_API_BASE = "https://khipus-ai-dev-chmrq-resource.openai.azure.com/openai/v1"
AZURE_OPENAI_CHAT_DEPLOYMENT = "gpt-4.1-mini"

In [12]:
base_url = AZURE_OPENAI_API_BASE.strip().rstrip("/") + "/"

url = urlparse(base_url)
if url.scheme != "https" or not url.netloc or url.path != "/openai/v1/":
    raise ValueError("AZURE_OPENAI_API_BASE debe ser el endpoint HTTPS que termina en /openai/v1/.")

if not AZURE_OPENAI_API_KEY.strip():
    raise ValueError("Completa AZURE_OPENAI_API_KEY en la celda de constantes antes de continuar.")

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY.strip(),
    base_url=base_url,
    timeout=60.0,
    max_retries=2,
)
print("Cliente configurado. La primera solicitud comprobará la conexión.")

Cliente configurado. La primera solicitud comprobará la conexión.


## Seleccionar el despliegue
En Azure, `model` recibe el **nombre del despliegue**. Esta práctica usa `gpt-4.1-mini`; puedes repetirla con `gpt-4.1`, también presente en tu proyecto.

Los parámetros de estas celdas están preparados para esos dos despliegues. `gpt-5`, `o4-mini` y `model-router` requieren revisar su compatibilidad antes de sustituirlos. No hace falta desplegar otro modelo para completar la actividad.

In [13]:
chat_deployment = AZURE_OPENAI_CHAT_DEPLOYMENT.strip()
if not chat_deployment:
    raise ValueError("AZURE_OPENAI_CHAT_DEPLOYMENT no puede estar vacía.")
print("Despliegue de texto:", chat_deployment)

Despliegue de texto: gpt-4.1-mini


## 1. Preparar una primera solicitud
El mensaje `system` define el papel del asistente. El mensaje `user` contiene la tarea concreta. Este ejemplo explica conceptos; no consulta páginas web ni información en tiempo real.

In [14]:
text_prompt = "¿Qué es un modelo de lenguaje grande y para qué se usa?"

messages = [
    {"role": "system", "content": "Eres un docente de inteligencia artificial. Responde en español con precisión y claridad."},
    {"role": "user", "content": text_prompt},
]

## 2. Generar y revisar el texto
`max_completion_tokens` limita la salida. `temperature` controla la variación de la generación en este modelo; no representa la veracidad de la respuesta.

In [15]:
response = client.chat.completions.create(
    model=chat_deployment,
    messages=messages,
    temperature=0.4,
    max_completion_tokens=600,
)

choice = response.choices[0]
text = choice.message.content
if not text:
    raise RuntimeError(
        f"El servicio no devolvió texto. Revisa el filtro o finish_reason={choice.finish_reason}."
    )
print(text)
print("\nMotivo de finalización:", choice.finish_reason)
if choice.finish_reason == "length":
    print("La respuesta alcanzó el límite. Reduce la extensión solicitada o ajusta el límite.")

Un modelo de lenguaje grande (en inglés, Large Language Model o LLM) es un tipo de modelo de inteligencia artificial entrenado con enormes cantidades de texto para comprender, generar y manipular lenguaje natural. Estos modelos utilizan arquitecturas de redes neuronales profundas, como los transformadores, y cuentan con millones o incluso miles de millones de parámetros que les permiten captar patrones complejos en el lenguaje.

**¿Para qué se usan?**

Los modelos de lenguaje grandes se emplean en diversas aplicaciones relacionadas con el procesamiento del lenguaje natural, tales como:

- Generación automática de texto coherente y contextual.
- Traducción automática entre idiomas.
- Resumen de documentos largos.
- Respuesta a preguntas y asistentes virtuales.
- Análisis de sentimientos y clasificación de texto.
- Completar frases o código.
- Conversación y chatbots.

En resumen, un modelo de lenguaje grande es una herramienta avanzada que permite a las máquinas entender y producir leng

## 3. Observar los tokens
Los tokens son las unidades de texto que procesa el modelo; no coinciden necesariamente con palabras. El uso ayuda a comparar solicitudes, pero no equivale por sí solo al costo monetario.

In [16]:
if response.usage is not None:
    print("Tokens de entrada:", response.usage.prompt_tokens)
    print("Tokens de salida:", response.usage.completion_tokens)
    print("Tokens totales:", response.usage.total_tokens)

Tokens de entrada: 41
Tokens de salida: 206
Tokens totales: 247


## 4. Mejorar el prompt
Mantén el modelo y la temperatura. Cambia solamente la instrucción del usuario para observar el efecto del público, el formato y las restricciones.

In [17]:
improved_prompt = (
    "Explica qué es un modelo de lenguaje grande a estudiantes que conocen Python "
    "pero no han estudiado inteligencia artificial. "
    "Responde en español con: una definición de hasta 50 palabras, "
    "dos ejemplos de uso y una limitación relacionada con las alucinaciones. "
    "Evita presentar el modelo como una fuente infalible."
)

improved_response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        messages[0],
        {"role": "user", "content": improved_prompt},
    ],
    temperature=0.4,
    max_completion_tokens=600,
)

improved_choice = improved_response.choices[0]
if not improved_choice.message.content:
    raise RuntimeError(f"Respuesta vacía: finish_reason={improved_choice.finish_reason}.")
print(improved_choice.message.content)
print("\nMotivo de finalización:", improved_choice.finish_reason)
if improved_choice.finish_reason == "length":
    print("La respuesta alcanzó el límite; revisa si está completa.")
if improved_response.usage is not None:
    print("Tokens totales:", improved_response.usage.total_tokens)

Un modelo de lenguaje grande es un programa entrenado con mucho texto para entender y generar lenguaje natural, como si fuera humano.  

Ejemplos de uso:  
1. Ayudar a escribir código en Python.  
2. Responder preguntas o resumir textos automáticamente.  

Limitación: A veces inventa información incorrecta (alucinaciones), por eso no siempre es fiable.

Motivo de finalización: stop
Tokens totales: 169


## Actividad del estudiante
- Identifica qué parte de cada respuesta cumple la instrucción del usuario.
- Explica por qué la segunda respuesta puede resultar más adecuada para el público indicado.
- Cambia el público a “personas sin experiencia en programación” y vuelve a ejecutar solo la segunda solicitud.
- Registra tus observaciones. Una única comparación no demuestra que un prompt siempre sea mejor.

**Observaciones:** escribe aquí tus conclusiones.